<a href="https://colab.research.google.com/github/Sanjaysanjay31/24B11DS095_DriveReady_OPC_2026/blob/main/Day_4/Startup_Marketing_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 4 — Tools, Structured Output, and Google Sheets

**Project:** AI Startup Marketing Agent  
**Primary Provider:** OpenRouter — NVIDIA Nemotron 3.5 Super VL (Free)  
**Fallback Provider:** Groq — GPT-OSS 120B  
**Database:** [Google Sheets Campaign Orders](https://docs.google.com/spreadsheets/d/1WryffEEH-N9TpWcwg8pvdaNU0XGzQWqBRS2Lnt0Ro90)  

On Day 3 our agent gained conversational memory. Today it gains **tools**: Python functions it calls autonomously to look up marketing service prices, check operating hours, validate campaign orders with Pydantic schemas, and record confirmed bookings directly into our live Google Sheets database.

In [1]:
!pip install -q langchain langchain-groq langchain-openrouter gspread google-auth python-dotenv

import os

# Load environment variables from .env locally or Colab Secrets
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

try:
    from google.colab import userdata
    colab_openrouter = userdata.get("OPENROUTER_API_KEY")
    colab_groq = userdata.get("GROQ_API_KEY") or userdata.get("agentic_ai_GROQ_API_KEY")
except Exception:
    colab_openrouter = None
    colab_groq = None

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY") or colab_openrouter
GROQ_API_KEY = os.getenv("GROQ_API_KEY") or colab_groq

from langchain_groq import ChatGroq
from langchain_openrouter import ChatOpenRouter

# Primary LLM: OpenRouter (NVIDIA Nemotron 3.5 Super VL) | Fallback: Groq (GPT-OSS 120B)
openrouter_llm = ChatOpenRouter(model="nvidia/nemotron-3.5-super-vl:free", temperature=0.2, api_key=OPENROUTER_API_KEY) if OPENROUTER_API_KEY else None
groq_llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0.2, max_retries=3, api_key=GROQ_API_KEY) if GROQ_API_KEY else None

if openrouter_llm and groq_llm:
    llm = openrouter_llm.with_fallbacks([groq_llm])
else:
    llm = openrouter_llm or groq_llm

# Google Sheets connection
ws = None
SPREADSHEET_ID = "1WryffEEH-N9TpWcwg8pvdaNU0XGzQWqBRS2Lnt0Ro90"

try:
    from google.colab import auth
    auth.authenticate_user()
    import gspread
    from google.auth import default
    creds, _ = default()
    gc = gspread.authorize(creds)
    sh = gc.open_by_key(SPREADSHEET_ID)
    ws = sh.sheet1
    print("Connected to Google Sheet in Colab:", sh.title, f"({sh.url})")
except Exception as e:
    try:
        import gspread
        gc = gspread.service_account(filename="google_service_account.json")
        sh = gc.open_by_key(SPREADSHEET_ID)
        ws = sh.sheet1
        print("Connected to Google Sheet via Service Account:", sh.title)
    except Exception:
        print("Notice: Running locally without Google credentials. Orders will be validated and logged locally.")

# Ensure headers exist in the sheet
if ws:
    try:
        headers = ["client_name", "startup_name", "service", "platform", "target_date", "total_price"]
        if not ws.get_all_values():
            ws.append_row(headers)
    except Exception:
        pass

print("Setup complete! Model and Google Sheets integration ready.")

Setup complete! Model and Google Sheets integration ready.


In [2]:
from typing import Literal
from pydantic import BaseModel, Field
from langchain.tools import tool

# Service catalogue and operating schedule for startup marketing
SERVICES = {
    "viral hook generation": 400,
    "short script writing": 700,
    "storyboard generation": 1200,
    "voiceover synthesis": 500,
    "auto subtitle karaoke": 300,
    "vertical video assembly": 1500,
    "seo metadata pack": 350
}

HOURS = {
    "monday": "9 am to 8 pm",
    "tuesday": "9 am to 8 pm",
    "wednesday": "9 am to 8 pm",
    "thursday": "9 am to 8 pm",
    "friday": "9 am to 8 pm",
    "saturday": "10 am to 6 pm",
    "sunday": "10 am to 2 pm"
}

class CampaignOrder(BaseModel):
    """A short-form video marketing campaign order placed by a founder."""
    client_name: str = Field(description="Founder's full or first name")
    startup_name: str = Field(description="Name of the startup product or company")
    service: Literal[
        "viral hook generation",
        "short script writing",
        "storyboard generation",
        "voiceover synthesis",
        "auto subtitle karaoke",
        "vertical video assembly",
        "seo metadata pack"
    ] = Field(description="Selected marketing service from the catalogue")
    platform: Literal["YouTube Shorts", "Instagram Reels", "TikTok", "All"] = Field(description="Target video platform")
    target_date: str = Field(description="Requested delivery or launch date as stated by the founder")

In [3]:
@tool
def get_service_price(service: str, qty: int = 1) -> str:
    """Get the total price in rupees for a marketing service from the catalogue.

    Args:
        service: Name of the service, e.g. 'viral hook generation', 'short script writing', or 'vertical video assembly'
        qty: Number of videos or service units requested
    """
    price = SERVICES.get(service.lower().strip())
    if price is None:
        available = ", ".join(SERVICES.keys())
        return f"'{service}' is not in our catalogue. Available services: {available}."
    total = price * qty
    return f"{qty} x {service} = Rs {total}"

@tool
def check_operating_hours(day: str) -> str:
    """Check the agency and rendering pipeline operating hours for a day of the week, e.g. 'sunday'."""
    hours = HOURS.get(day.lower().strip())
    return f"Support & video pipeline open {hours} on {day}." if hours else f"'{day}' is not a valid day of the week."

@tool
def log_campaign_order(client_name: str, startup_name: str, service: str, platform: str, target_date: str) -> str:
    """Save a confirmed campaign order into the Google Sheets database.
    Use ONLY after confirming the client's name, startup name, service, platform, and delivery date."""
    try:
        order = CampaignOrder(
            client_name=client_name.strip(),
            startup_name=startup_name.strip(),
            service=service.lower().strip(),
            platform=platform,
            target_date=target_date.strip()
        )
    except Exception as e:
        return f"Could not validate order: {e}. Ask the founder to clarify missing or invalid details."

    total_price = SERVICES[order.service]

    # Append to Google Sheet
    if ws:
        try:
            ws.append_row([
                order.client_name,
                order.startup_name,
                order.service,
                order.platform,
                order.target_date,
                total_price
            ])
            return f"Logged to Google Sheet: {order.service} for {order.client_name} ({order.startup_name}) targeting {order.platform} by {order.target_date}, Total: Rs {total_price}."
        except Exception as err:
            return f"Sheet error: {err}. Order validated locally for {order.client_name}, Total: Rs {total_price}."
    else:
        try:
            import csv
            csv_path = "campaign_orders.csv"
            file_exists = os.path.exists(csv_path)
            with open(csv_path, "a", newline="", encoding="utf-8") as f:
                writer = csv.writer(f)
                if not file_exists:
                    writer.writerow(["client_name", "startup_name", "service", "platform", "target_date", "total_price"])
                writer.writerow([order.client_name, order.startup_name, order.service, order.platform, order.target_date, total_price])
        except Exception:
            pass
        return f"Order confirmed and logged locally: {order.service} for {order.client_name} ({order.startup_name}) on {order.target_date}, Total: Rs {total_price}."

In [4]:
from langchain.agents import create_agent

SYSTEM = """ROLE:   You are the AI Startup Marketing Agent for early-stage founders creating 9:16 vertical videos.
FACTS:  We produce high-retention faceless videos for YouTube Shorts and Instagram Reels.
RULES:  Always use get_service_price for pricing and check_operating_hours for timings; never guess them.
        Before calling log_campaign_order, ensure you have: client name, startup name, service, platform, and target date. If any is missing, ask politely.
        After logging the order, confirm the details back with the total price.
FORMAT: Under 60 words, encouraging, practical, and clear."""

agent = create_agent(model=llm, tools=[get_service_price, check_operating_hours, log_campaign_order], system_prompt=SYSTEM)

history = []
def chat(text: str) -> str:
    global history
    try:
        result = agent.invoke({"messages": history + [{"role": "user", "content": text}]})
        history = result["messages"]
        return history[-1].content
    except Exception as e:
        print("Agent error:", e)
        return "I encountered a brief issue. Please try your request again."

print("Agent:", chat("Are you open on Sunday? How much is vertical video assembly?"))
print("Agent:", chat("I'm Sanjay from DevPulse. Please book short script writing for YouTube Shorts by 15 Dec."))

Agent: On Sunday, our support and video rendering pipeline is open from 10 am to 2 pm. Vertical video assembly is Rs 1500 per video.
Agent: I have confirmed and logged your order to our Google Sheet: short script writing for Sanjay (DevPulse) targeting YouTube Shorts by 15 Dec, Total: Rs 700.
